# Примеры метрик `CalculateMetrics`

Каждый блок показывает, что метрика умеет различать и где она слепнет. Аномалия — метка 0, штатная точка — метка больше 0. Ряд — кортеж (время, долгота, широта, метка).

Сначала карта исходного трека и пунктирного эталона через аномальный разрыв, затем — разметка модели (две фигуры подряд). Точки синтетические, район Владивостока выбран только чтобы была читаемая подложка. Подложка качается из интернета; тайлы кэшируются в ~/.cache/statistical_researchs/map_tiles. Без сети подложка подставится из кэша, если он уже был; иначе рисуются только точки.

**Обозначения**: ряд prediction $P$ и ground truth $G$ — кортежи $(t_i, \mathrm{lon}_i, \mathrm{lat}_i, y_i)$. Аномалия: $y_i=0$, штатное состояние: $y_i>0$. Бинарно $a_i=\mathbf{1}[y_i=0]$, $\hat a_i=\mathbf{1}[\hat y_i=0]$. Геодезическое расстояние $d_{\mathrm{geo}}$ — по большому кругу, $R=6371\,\mathrm{км}$.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import importlib
import sys
from pathlib import Path

import numpy as np

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "app" / "metrics" / "calculate_metrics.py").exists():
        sys.path.insert(0, str(candidate))
        break

import app.evaluation.trajectory_metrics as calculate_metrics_mod
import app.viz.track_osm as plot_track_maps_mod

importlib.reload(calculate_metrics_mod)
importlib.reload(plot_track_maps_mod)

CalculateMetrics = calculate_metrics_mod.CalculateMetrics
METRIC_TABLE_ROWS = calculate_metrics_mod.METRIC_TABLE_ROWS
METRIC_LABELS = {key: label for key, label in METRIC_TABLE_ROWS}
print_metrics_table = calculate_metrics_mod.print_metrics_table
format_metric_value = calculate_metrics_mod.format_metric_value
sample_track = calculate_metrics_mod.demo_track_with_delay
plot_track_maps = plot_track_maps_mod.plot_track_maps


def metric_rows(*keys: str):
    """Подписи столбцов как в example_var_1 / METRIC_TABLE_ROWS."""
    return [(key, METRIC_LABELS.get(key, key)) for key in keys]


def course(count=12, step=0.004):
    """Прямой ход на северо-восток. Шаг около 500 м."""
    index = np.arange(count, dtype=float)
    return 43.04 + step * index, 131.87 + step * index


def as_series(latitude, longitude, labels, predicted):
    """Собирает ground truth и prediction на одних координатах."""
    time = np.arange(len(latitude), dtype=float)
    latitude = np.asarray(latitude, dtype=float)
    longitude = np.asarray(longitude, dtype=float)
    ground_truth = (time, longitude, latitude, np.asarray(labels, dtype=float))
    prediction = (time, longitude.copy(), latitude.copy(), np.asarray(predicted, dtype=float))
    return prediction, ground_truth


_events_from_mask = calculate_metrics_mod._events_from_mask
_affiliation_zones = calculate_metrics_mod._affiliation_zones

AFFILIATION_FIELDS = (
    "point_f1",
    "affiliation_precision",
    "affiliation_recall",
    "affiliation_f1",
)


def print_affiliation_zones(ground_truth, title="Зоны affiliation"):
    """Печатает интервалы gt-событий и ширину зоны на оси времени."""
    time = np.asarray(ground_truth[0], dtype=float)
    gt_anomaly = np.equal(ground_truth[3], 0.0)
    events, t_range = _events_from_mask(time, gt_anomaly)
    zones = _affiliation_zones(events, t_range)
    print(f"  {title}")
    print(f"  Диапазон ряда: [{t_range[0]:.2f}, {t_range[1]:.2f})")
    for index, (event, zone) in enumerate(zip(events, zones), start=1):
        width = zone[1] - zone[0]
        print(
            f"  Событие {index}: gt [{event[0]:.2f}, {event[1]:.2f}), "
            f"зона [{zone[0]:.2f}, {zone[1]:.2f}), ширина {width:.2f}"
        )
    return events, zones


def show_case(
    title,
    prediction,
    ground_truth,
    fields=None,
    scores=None,
    full_table: bool = False,
    plot: bool = True,
):
    """Считает метрики, печатает таблицу и при plot=True рисует карты."""
    metrics = CalculateMetrics.calculate_all_metrics(prediction, ground_truth, scores)
    if full_table:
        print_metrics_table(metrics, title=title, prefix="  ")
    else:
        if not fields:
            raise ValueError("Укажите fields или full_table=True")
        print_metrics_table(metrics, title=title, rows=metric_rows(*fields), prefix="  ")
    print()
    if plot:
        plot_track_maps(prediction, ground_truth, title=title)
    return metrics

## Полный вызов `calculate_all_metrics`

In [ ]:
reference_prediction, reference_truth, reference_scores = sample_track()
reference_metrics = show_case(
    "Эталон: задержка детекции и ложная тревога (все метрики)",
    reference_prediction,
    reference_truth,
    scores=reference_scores,
    full_table=True,
)

perfect_prediction = (
    reference_truth[0],
    reference_truth[1],
    reference_truth[2],
    reference_truth[3].copy(),
)
perfect_metrics = CalculateMetrics.calculate_all_metrics(
    perfect_prediction,
    reference_truth,
    reference_scores,
)
print("Сравнение с идеальной маской (те же координаты и score):")
print_metrics_table(
    perfect_metrics,
    title="Идеальная разметка",
    prefix="  ",
)
print(
    "  Distance loss, м:",
    format_metric_value("distance_loss", perfect_metrics["distance_loss"]),
    "  DLR:",
    format_metric_value("distance_loss_ratio", perfect_metrics["distance_loss_ratio"]),
)

## Point-wise F1

**Формула.** Баланс точности и полноты по точкам; время и координаты не входят.

$$\mathrm{Precision}=\frac{TP}{TP+FP}, \qquad \mathrm{Recall}=\frac{TP}{TP+FN}, \qquad F_1=\frac{2TP}{2TP+FP+FN}.$$

**Пояснение.** Каждая точка ряда — отдельное бинарное решение. Сдвиг маски на один индекс даёт и пропуск, и ложную тревогу, поэтому $F_1$ падает резко.

**Плюс.** Лишняя тревога видна сразу. На участке 4–7 четыре аномалии найдены верно, точка 10 помечена некорректно. Precision = 4/5 = 0.8.

**Минус.** Тот же участок найден на шаг позже: помечены точки 5–8 вместо 4–7. Point-wise F1 падает до 0.75 из-за одной пропущенной и одной лишней точки.


In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0

found = labels.copy()
found[10] = 0
prediction, ground_truth = as_series(latitude, longitude, labels, found)
show_case(
    "Point-wise: одна лишняя тревога на точке 10",
    prediction,
    ground_truth,
    ["point_precision", "point_recall", "point_f1"],
)

shifted = np.ones(12)
shifted[5:9] = 0
prediction, ground_truth = as_series(latitude, longitude, labels, shifted)
show_case(
    "Point-wise: событие найдено, но на шаг позже",
    prediction,
    ground_truth,
    ["point_precision", "point_recall", "point_f1", "affiliation_f1"],
)

## PR-AUC

**Формула.** Площадь под кривой precision–recall по непрерывному anomaly score $s_i$ (больше $s_i$ — сильнее аномалия). Для порога $\tau$, $\hat a_i(\tau)=\mathbf{1}[s_i\ge\tau]$:

$$\mathrm{Precision}(\tau)=\frac{TP(\tau)}{TP(\tau)+FP(\tau)}, \qquad \mathrm{Recall}(\tau)=\frac{TP(\tau)}{TP(\tau)+FN(\tau)},$$

$$PR\text{-}AUC=\int_0^1 \mathrm{Precision}(r)\,dr.$$

Узлы — уровни $s_i$ по убыванию; интеграл — трапециями. Без вектора `anomaly_scores` метрика не определена (`nan`).

**Пояснение.** Оценивает **ранжирование** точек по score, а не одну выбранную маску. Point F1 зависит от порога бинаризации; PR-AUC — нет, если порядок score верный.

**Плюс.** Score один и тот же. Порог 0.9 оставляет только две самые сильные аномалии, point-wise F1 = 0.67. Порог 0.5 забирает все четыре, F1 = 1. PR-AUC в обоих случаях равен 1: ранжирование верное и от выбранного порога не зависит.

**Минус.** Тот же идеальный порядок score стоит и на прямом треке, и на петле, которую модель целиком пропустила. PR-AUC снова 1. Геодезический RMSE при этом 0 м против примерно 4.8 км: метрика не видит, куда уехали точки. Если score не передать, `pr_auc` равен `nan`, хотя point-wise F1 по маске считается.


In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
scores = np.full(12, 0.12)
scores[4:8] = [0.58, 0.74, 0.91, 0.96]
scores[10] = 0.47

for threshold, caption in ((0.9, "жёсткий порог 0.9"), (0.5, "мягкий порог 0.5")):
    predicted = np.where(scores >= threshold, 0.0, 1.0)
    prediction, ground_truth = as_series(latitude, longitude, labels, predicted)
    show_case(
        f"PR-AUC: {caption}",
        prediction,
        ground_truth,
        ["point_f1", "pr_auc"],
        scores=scores,
    )

missed = np.ones(12)
prediction, ground_truth = as_series(latitude, longitude, labels, missed)
without_scores = CalculateMetrics.calculate_all_metrics(prediction, ground_truth)
print(
    "Та же мягкая маска без score: pr_auc =",
    format_metric_value("pr_auc", without_scores["pr_auc"]),
)
print()

show_case(
    "PR-AUC: пропуск на прямой хорде, score идеальный",
    prediction,
    ground_truth,
    ["point_f1", "pr_auc", "geodesic_rmse"],
    scores=scores,
)

loop = latitude.copy()
loop[4:8] += [0.015, 0.045, 0.07, 0.02]
prediction, ground_truth = as_series(loop, longitude, labels, missed)
show_case(
    "PR-AUC: та же модель пропускает петлю",
    prediction,
    ground_truth,
    ["point_f1", "pr_auc", "geodesic_rmse"],
    scores=scores,
)

## Affiliation F1

**Формула** (event-level по времени). Ось времени делится на зоны $I_j$ вокруг каждого истинного события $gt_j$. По зоне считаются локальные $p_j$, $r_j$ из временных расстояний между интервалами $pred$ и $gt$ (survival function; координаты не используются):

$$\mathrm{Precision}=\frac{1}{|S|}\sum_{j\in S} p_j, \qquad \mathrm{Recall}=\frac{1}{m}\sum_{j=1}^{m} r_j, \qquad F_1=2\frac{\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}.$$

$S$ — события, куда попало хотя бы одно предсказание; $m$ — число всех истинных событий. Непрерывный участок $\{i:y_i=0\}$ задаёт полуинтервал $[t_{\mathrm{start}}, t_{\mathrm{end}+1})$.

**Пояснение.** Штраф за сдвиг интервала по времени обычно **мягче**, чем у point-wise F1. Ширина зоны $I_j$ зависит от расстояния до соседних событий и до края ряда.

**Плюс.** Сдвиг на один шаг, уже виденный выше. Point-wise F1 = 0.75, affiliation F1 ≈ 0.94. Для affiliation предсказанный интервал почти совпадает с истинным.

**Минус.** Модель помечает аномалией весь трек. Point-wise precision = 4/12 ≈ 0.33 и F1 = 0.5. Affiliation F1 ≈ 0.71: сплошная заливка для неё близка к случайному угадыванию внутри зоны события, а не к полному провалу. Редкость аномалий она штрафует слабее, чем точечная precision.

Далее — как **ширина зоны** (до соседнего события или края ряда) влияет на affiliation при том же сдвиге маски или ложной тревоге. «Окно» только на оси времени, не в метрах на карте.


In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
shifted = np.ones(12)
shifted[5:9] = 0
prediction, ground_truth = as_series(latitude, longitude, labels, shifted)
show_case(
    "Affiliation: сдвиг на шаг почти не штрафуется",
    prediction,
    ground_truth,
    ["point_f1", "affiliation_precision", "affiliation_recall", "affiliation_f1"],
)

everything = np.zeros(12)
prediction, ground_truth = as_series(latitude, longitude, labels, everything)
show_case(
    "Affiliation: аномалией помечен весь трек",
    prediction,
    ground_truth,
    ["point_precision", "point_f1", "affiliation_precision", "affiliation_recall", "affiliation_f1"],
)

### Ширина зон affiliation

**Формула границы зоны.** Для событий $gt_1,\ldots,gt_m$ левая и правая граница $I_j$ — середины пауз между соседними событиями (у краёв ряда — до $t_0$ и $t_N$). Ширина $|I_j|$ задаёт масштаб штрафа в локальных $p_j$, $r_j$.

**Пояснение.** Длинный ряд с одним событием даёт **широкую** зону: тот же сдвиг маски на один шаг обычно **меньше** бьёт по affiliation F1, чем на коротком ряде (point F1 при этом одинаков). **Близкие** события дают **узкие** зоны — предсказание на стыке сильнее режет precision и recall. Ложная тревога внутри зоны, но вне истинного интервала, снижает **affiliation precision**; recall истинного события может остаться 1.


In [ ]:
def labels_with_events(count: int, *spans: tuple[int, int]):
    """spans — полуинтервалы индексов [start, stop) с меткой аномалии 0."""
    labels = np.ones(count)
    for start, stop in spans:
        labels[start:stop] = 0.0
    return labels


def compare_shift_one_event(short_count=12, long_count=50, event_len=4):
    """Один и тот же сдвиг маски на 1 шаг при коротком и длинном ряде."""
    rows = []
    for count, start, caption in (
        (short_count, 4, f"короткий ряд (N={short_count})"),
        (long_count, 18, f"длинный ряд (N={long_count})"),
    ):
        labels = labels_with_events(count, (start, start + event_len))
        shifted = np.ones(count)
        shifted[start + 1 : start + event_len + 1] = 0.0
        lat, lon = course(count)
        prediction, ground_truth = as_series(lat, lon, labels, shifted)
        print_affiliation_zones(ground_truth, caption)
        metrics = show_case(
            f"Сдвиг на 1 шаг: {caption}",
            prediction,
            ground_truth,
            AFFILIATION_FIELDS,
            plot=False,
        )
        rows.append((caption, metrics))
        print()
    print("Сводка: point F1 совпадает, affiliation F1 на длинном ряде обычно выше:")
    for caption, metrics in rows:
        print(
            f"  {caption}: point F1 {format_metric_value('point_f1', metrics['point_f1'])}, "
            f"affiliation F1 {format_metric_value('affiliation_f1', metrics['affiliation_f1'])}"
        )


compare_shift_one_event()

for count, start in ((12, 4), (50, 18)):
    labels = labels_with_events(count, (start, start + 4))
    predicted = labels.copy()
    predicted[0] = 0.0
    lat, lon = course(count)
    prediction, ground_truth = as_series(lat, lon, labels, predicted)
    print_affiliation_zones(ground_truth, f"ложная тревога t=0, N={count}")
    show_case(
        f"Ложная тревога в начале ряда (N={count})",
        prediction,
        ground_truth,
        AFFILIATION_FIELDS,
        plot=False,
    )

count = 24
labels = labels_with_events(count, (4, 8), (9, 13))
shifted = np.ones(count)
shifted[5:9] = 0.0
lat, lon = course(count)
prediction, ground_truth = as_series(lat, lon, labels, shifted)
print_affiliation_zones(ground_truth, "два события с паузой 1 шаг")
show_case(
    "Два близких события: pred сдвинут и пересекает границу зон",
    prediction,
    ground_truth,
    AFFILIATION_FIELDS,
)

count = 40
labels = labels_with_events(count, (4, 8), (28, 32))
found_first = np.ones(count)
found_first[4:7] = 0.0
lat, lon = course(count)
prediction, ground_truth = as_series(lat, lon, labels, found_first)
print_affiliation_zones(ground_truth, "два далёких события")
show_case(
    "Два далёких события: детекция только у первого",
    prediction,
    ground_truth,
    AFFILIATION_FIELDS,
)

## Detection delay

**Формула.** Для каждого истинного интервала $S_k=\{i:y_i=0\}$:

$$t_k^{\mathrm{start}}=\min_{i\in S_k} t_i, \qquad t_k^{\mathrm{detect}}=\min\{t_i: i\in S_k,\ \hat y_i=0\}, \qquad \mathrm{Delay}_k=t_k^{\mathrm{detect}}-t_k^{\mathrm{start}}.$$

Если внутри $S_k$ нет предсказанной аномалии, событие **undetected** (в среднее не подставляется 0). По обнаруженным событиям — mean, median, p95; также счётчики Events / Detected / Undetected.

**Пояснение.** Метрика про **скорость** первого срабатывания внутри эпизода, а не про точное совпадение всех точек (это point F1).

**Плюс.** Оба детектора находят 3 из 5 точек, point-wise F1 у обоих 0.75, affiliation тоже совпадает. Ранний начинает с первой точки участка, поздний — только с третьей. Средняя задержка 0 против 2. Эту разницу F1 не видит.

**Минус.** Три отдельных события. Модель сразу ловит первую точку первого и пропускает два других. Средняя задержка равна 0, потому что в неё попало единственное обнаруженное событие. Пропуски лежат в `n_undetected = 2`; point-wise recall при этом 0.11.


In [ ]:
latitude, longitude = course(14)
labels = np.ones(14)
labels[4:9] = 0
early = np.ones(14)
early[4:7] = 0
late = np.ones(14)
late[6:9] = 0
delay_fields = (
    "point_f1",
    "affiliation_f1",
    "detection_delay_mean",
    "detection_delay_median",
    "detection_delay_p95",
    "detection_delay_n_events",
    "detection_delay_n_detected",
    "detection_delay_n_undetected",
)

prediction, ground_truth = as_series(latitude, longitude, labels, early)
show_case("Delay: детекция с первой точки", prediction, ground_truth, delay_fields)
prediction, ground_truth = as_series(latitude, longitude, labels, late)
show_case("Delay: те же три точки, но с опозданием", prediction, ground_truth, delay_fields)

latitude, longitude = course(18, step=0.0035)
labels = np.ones(18)
labels[2:5] = 0
labels[8:11] = 0
labels[14:17] = 0
only_first = np.ones(18)
only_first[2] = 0
prediction, ground_truth = as_series(latitude, longitude, labels, only_first)
show_case(
    "Delay: мгновенно найдено одно событие из трёх",
    prediction,
    ground_truth,
    delay_fields,
)

## Geodesic RMSE

**Формула.** Только **пропущенные** аномалии (false negative). Опорная точка на хорде между штатными соседями разрыва; $d_i=d_{\mathrm{geo}}\bigl((\mathrm{lat}_i,\mathrm{lon}_i),(\varphi_i^*,\lambda_i^*)\bigr)$:

$$S_{FN}=\{i:y_i=0\land\hat y_i\neq 0\}, \qquad RMSE_d=\sqrt{\frac{1}{|S_{FN}\cap V|}\sum_{i\in S_{FN}\cap V} d_i^2}.$$

$V$ — точки разрыва с обоими штатными соседями. Обнаруженная аномалия в сумму не входит; при пустом $S_{FN}$ значение 0.

**Пояснение.** Штрафует **где** пропущенная аномалия ушла от штатной хорды, а не факт ложной тревоги.

**Плюс.** Обе модели пропускают один и тот же участок, point-wise F1 у обеих 0. Сдвиг примерно на 200 м даёт RMSE около 222 м. Петля в сторону города — около 4.8 км. Точечная метрика эти пропуски не различает.

**Минус.** Та же петля, но модель помечает её аномалией. RMSE становится 0: штрафуются только false negative. На левой панели петля никуда не делась, пространственного штрафа уже нет.


In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
missed = np.ones(12)
fields = ["point_f1", "geodesic_rmse", "hausdorff_distance"]

small = latitude.copy()
small[4:8] += 0.002
prediction, ground_truth = as_series(small, longitude, labels, missed)
show_case("RMSE: пропущенный сдвиг около 200 м", prediction, ground_truth, fields)

loop = latitude.copy()
loop[4:8] += [0.015, 0.045, 0.07, 0.02]
prediction, ground_truth = as_series(loop, longitude, labels, missed)
show_case("RMSE: пропущена петля в несколько километров", prediction, ground_truth, fields)

prediction, ground_truth = as_series(loop, longitude, labels, labels)
show_case("RMSE: та же петля обнаружена, штраф обнулён", prediction, ground_truth, fields)

## Hausdorff distance

**Формула.** Те же $d_i$ и множество $S_{FN}\cap V$, что у geodesic RMSE:

$$H_{FN}=\max_{i\in S_{FN}\cap V} d_i.$$

При пустом $S_{FN}$ — 0. Это **максимум** ошибки пропуска, а не симметричное Hausdorff между множествами точек.

**Пояснение.** Одна «худшая» пропущенная точка задаёт всю метрику; RMSE усредняет по всем FN.

**Плюс.** Четыре пропущенные точки сдвинуты одинаково: RMSE и Хаусдорф совпадают, около 3 км. Если три сдвига малы, а четвёртая точка уходит на 6 км, RMSE остаётся около 3 км, Хаусдорф вырастает до 6 км. Он равен худшей точке.

**Минус.** Худшая точка та же, но в одном треке далеко ушла только она, а в другом — все четыре. Хаусдорф одинаковый, около 6 км. RMSE при этом около 3 км против 6 км. Одна выбросная точка задаёт всю метрику.


In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
missed = np.ones(12)
fields = ["geodesic_rmse", "hausdorff_distance"]

even = latitude.copy()
even[4:8] += 0.027
prediction, ground_truth = as_series(even, longitude, labels, missed)
show_case("Hausdorff: четыре точки сдвинуты одинаково", prediction, ground_truth, fields)

spike = latitude.copy()
spike[4:8] += [0.004, 0.004, 0.004, 0.055]
prediction, ground_truth = as_series(spike, longitude, labels, missed)
show_case("Hausdorff: три точки рядом, одна далеко", prediction, ground_truth, fields)

far = latitude.copy()
far[4:8] += 0.055
prediction, ground_truth = as_series(far, longitude, labels, missed)
show_case("Hausdorff: далеко ушли все четыре точки", prediction, ground_truth, fields)

## Distance loss

**Формула.** Длина ломаной $L(T)=\sum d_{\mathrm{geo}}(p_i,p_{i+1})$ по выбранным точкам:

$$DL=|L_{\mathrm{pred}}-L_{\mathrm{ref}}|, \qquad DLR=\frac{DL}{L_{\mathrm{ref}}}.$$

$L_{\mathrm{ref}}$ — по $G$: штатные точки и интерполяция через внутренние разрывы. $L_{\mathrm{pred}}$ — по точкам с $\hat y_i\neq 0$; исключённые аномалии выбрасываются из штатного маршрута модели.

**Пояснение.** Описывает, насколько **длинным** модель оставила штатный путь, а не исходную геометрию всего трека.

**Плюс.** Обе модели пропускают участок целиком, point-wise F1 = 0. Сдвиг почти не меняет длину: отношение около 0.03. Петля, оставленная в штатном маршруте, делает его почти в четыре раза длиннее эталона, отношение около 2.9.

**Минус.** Та же петля обнаружена и выброшена из штатного маршрута. Чёрная линия на правой панели идёт по хорде, distance loss равен 0. Метрика описывает путь, который классификатор счёл штатным, а не исходную геометрию трека.


In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
missed = np.ones(12)
fields = ["point_f1", "distance_loss", "distance_loss_ratio", "geodesic_rmse"]

bump = latitude.copy()
bump[4:8] += 0.004
prediction, ground_truth = as_series(bump, longitude, labels, missed)
show_case("Distance loss: короткий сдвиг почти не удлиняет путь", prediction, ground_truth, fields)

loop = latitude.copy()
loop[4:8] += [0.02, 0.06, 0.09, 0.03]
prediction, ground_truth = as_series(loop, longitude, labels, missed)
show_case("Distance loss: петля принята за штатный ход", prediction, ground_truth, fields)

prediction, ground_truth = as_series(loop, longitude, labels, labels)
show_case("Distance loss: петля исключена из штатного маршрута", prediction, ground_truth, fields)